# Unit 8 · Decision trees

E8 and E3383 both have 0% exclamation marks. Unit 6 gave both about a 31.10% Spam score. Their recorded labels differ. This unit adds a second measurement, the longest consecutive run of uppercase letters, and asks which questions a tree can learn from these inputs.

Use the original Unit 6 training, validation, and test emails. The teaching fixture contains the two source measurements and the fitted reference models. The row IDs refer to lines in the original UCI data.

[Open the lesson](https://sharanharsoor.github.io/ml-learning-lab/ml-foundations/#unit-8) · [UCI Spambase](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import json
import os
from pathlib import Path
from urllib.request import urlopen

url = (
    "https://raw.githubusercontent.com/sharanharsoor/ml-learning-lab/"
    "main/labs/ml-foundations/data/unit-8.json"
)
if local_path := os.getenv("ML_FOUNDATIONS_TREE_FIXTURE"):
    fixture = json.loads(Path(local_path).read_text())
else:
    with urlopen(url, timeout=20) as response:
        fixture = json.load(response)

rows = fixture["rows"]
features = {feature["key"]: feature["label"] for feature in fixture["features"]}
labels = {0: "Legitimate", 1: "Spam"}
validation_by_id = {row["id"]: row for row in rows["validation"]}
for split, records in rows.items():
    print(f"{split.capitalize():<10} {len(records):>5,} emails")
for email in ("E8", "E3383"):
    row = validation_by_id[email]
    print(f'{email:>5} | ! {row["x"]:.3f}% | longest run {row["uppercase"]} | {labels[row["y"]]}')

## Follow a fitted tree

At each question, go left when the feature is at or below the cutoff. Go right otherwise. A leaf predicts its training majority. You can change the email and depth below, then compare the browser path.

The Spam share at a leaf is its recorded Spam count divided by its training count. A share of 50% or more predicts Spam, including when the class counts tie.

In [ ]:
def path_to_leaf(row, root):
    path = []
    node = root
    while "feature" in node:
        feature = node["feature"]
        value = row[feature]
        threshold = node["threshold"]
        side = "left" if value <= threshold else "right"
        path.append((node, value, side))
        node = node[side]
    return path, node

trace_depth = 2  # Try 1, 2, 3, or 4.
email = validation_by_id["E17"]  # Try E8, E3383, E3027, E20, or E54.
root = fixture["trees"][str(trace_depth)]["root"]
path, leaf = path_to_leaf(email, root)
print(f'{email["id"]} | recorded {labels[email["y"]]}')
for node, value, side in path:
    relation = "≤" if side == "left" else ">"
    print(f'{features[node["feature"]]} | {value:g} {relation} {node["threshold"]:g} → {side}')
print(f'Leaf has {leaf["spam"]} Spam and {leaf["legitimate"]} Legitimate training emails.')
print(f'Spam share = {leaf["spam"]} ÷ {leaf["n"]} = {leaf["score"]:.2%}')
print("Predict", labels[leaf["prediction"]])

## Score a candidate split on the training rows

Gini impurity is **1 − p(Spam)² − p(Legitimate)²**. Weight the two child impurities by their shares of the **current parent**, then subtract that weighted result from the parent impurity. At a deeper node, only emails that reached that node count.

The display below routes the actual training records and recalculates both candidate cuts at each worked node.

In [ ]:
def gini(records):
    spam = sum(row["y"] for row in records)
    n = len(records)
    legitimate = n - spam
    return 1 - (spam / n) ** 2 - (legitimate / n) ** 2

def reaches(row, root, node_id):
    path, leaf = path_to_leaf(row, root)
    return any(node["id"] == node_id for node, _, _ in path) or leaf["id"] == node_id

deep_root = fixture["trees"]["4"]["root"]
for worked in fixture["splitExamples"]:
    node_id = worked["nodeId"]
    parent = [row for row in rows["train"] if reaches(row, deep_root, node_id)]
    n = len(parent)
    print()
    print(f'{node_id}: {n} training emails | parent Gini = {gini(parent):.6f}')
    for candidate in worked["candidates"]:
        feature, threshold = candidate["feature"], candidate["threshold"]
        left = [row for row in parent if row[feature] <= threshold]
        right = [row for row in parent if row[feature] > threshold]
        weighted = (len(left) / n) * gini(left) + (len(right) / n) * gini(right)
        gain = gini(parent) - weighted
        print(f'  {features[feature]} ≤ {threshold:g}')
        print(f'  ({len(left)}/{n}) × {gini(left):.6f} + ({len(right)}/{n}) × {gini(right):.6f} = {weighted:.6f}')
        print(f'  Gain = {gini(parent):.6f} − {weighted:.6f} = {gain:.6f}')

## See the tree and the regions it creates

The highlighted branch connects the email to its prediction. Each split makes an axis-aligned cut inside the region belonging to its current node. Change the depth or email in the trace cell, then run this cell again. The plot zooms in for readability. The fit uses the full training feature ranges.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, (tree_ax, region_ax) = plt.subplots(1, 2, figsize=(max(13, 4 + 2 ** trace_depth), 5.4), layout="constrained")
walked = {node["id"] for node, _, _ in path} | {leaf["id"]}
class_colors = {0: "#bcdcd8", 1: "#f4dcca"}

def draw_node(node, lo, hi, level=0, parent=None):
    x = (lo + hi) / 2
    y = -level
    highlight = node["id"] in walked
    if parent:
        tree_ax.plot([parent[0], x], [parent[1], y], color="#664d87" if highlight else "#adbab1", lw=2)
    if "feature" in node:
        short = "!" if node["feature"] == "x" else "Caps run"
        content = f'{short} ≤ {node["threshold"]:g}'
    else:
        content = labels[node["prediction"]]
    content += f'{chr(10)}{node["spam"]} S / {node["legitimate"]} L'
    tree_ax.text(x, y, content, ha="center", va="center", fontsize=8 if trace_depth > 2 else 10,
                 bbox={"boxstyle": "round,pad=.28", "facecolor": "#f3eef9" if highlight else "#f6f8f6",
                       "edgecolor": "#664d87" if highlight else "#bbc5bc"}, zorder=3)
    if "feature" in node:
        draw_node(node["left"], lo, x, level + 1, (x, y))
        draw_node(node["right"], x, hi, level + 1, (x, y))

def draw_region(node, x0, x1, y0, y1):
    if x0 >= x1 or y0 >= y1:
        return
    if "feature" not in node:
        region_ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0,
                                      facecolor=class_colors[node["prediction"]], edgecolor="white"))
    elif node["feature"] == "x":
        cut = min(x1, max(x0, node["threshold"]))
        draw_region(node["left"], x0, cut, y0, y1)
        draw_region(node["right"], cut, x1, y0, y1)
    else:
        cut = min(y1, max(y0, node["threshold"]))
        draw_region(node["left"], x0, x1, y0, cut)
        draw_region(node["right"], x0, x1, cut, y1)

draw_node(root, 0.0, 1.0)
tree_ax.set(xlim=(-0.04, 1.04), ylim=(-trace_depth - .45, .45),
            title=f"Depth limit {trace_depth} · Path of {email['id']}")
tree_ax.axis("off")
xmax, ymax = max(2, email["x"] + .1), max(125, email["uppercase"] + 5)
draw_region(root, 0, xmax, 0, ymax)
for actual, color in [(0, "#246e64"), (1, "#ac4d26")]:
    group = [row for row in rows["validation"] if row["y"] == actual]
    region_ax.scatter([row["x"] for row in group], [row["uppercase"] for row in group], color=color,
                      s=11, alpha=.28, label="Recorded " + labels[actual])
region_ax.scatter(email["x"], email["uppercase"], marker="*", s=155, color="#664d87",
                  edgecolor="white", linewidth=.8, label=email["id"], zorder=4)
region_ax.set(xlim=(0, xmax), ylim=(0, ymax), xlabel="Exclamation marks (% of characters)",
              ylabel="Longest uppercase run", title="Leaf predictions · Teal Legitimate, orange Spam")
region_ax.legend(loc="upper right", fontsize=8)
plt.show()

## Choose depth on validation, then check the held-out emails

Print training and validation accuracy before selecting depth. If depths tie on validation accuracy, choose the shallower one. The test comparison uses the original Unit 6 test emails. Both classifiers use the two same measurements. The tree uses its leaf Spam share and the logistic model uses its fitted Spam probability. Both flag Spam at a score of 0.50 or higher.

In [ ]:
for depth in range(1, 5):
    fitted = fixture["trees"][str(depth)]
    print(f'Depth {depth} | train {fitted["train"]["accuracy"]:.2%} | validation {fitted["validation"]["accuracy"]:.2%}')

selected_depth = max(range(1, 5), key=lambda depth: fixture["trees"][str(depth)]["validation"]["accuracy"])
assert selected_depth == fixture["selectedDepth"]
print("Depth chosen using validation only:", selected_depth)

In [ ]:
def confusion(records, predict):
    counts = dict.fromkeys(["tp", "fp", "fn", "tn"], 0)
    cells = {(1, 1): "tp", (0, 1): "fp", (1, 0): "fn", (0, 0): "tn"}
    for row in records:
        counts[cells[(row["y"], predict(row))]] += 1
    return counts

tree_root = fixture["trees"][str(selected_depth)]["root"]
logistic = fixture["logistic"]

def tree_predict(row):
    return path_to_leaf(row, tree_root)[1]["prediction"]

def logistic_predict(row):
    z = logistic["intercept"] + sum(logistic["weights"][key] * row[key] for key in features)
    return int(z >= 0)  # Sigmoid reaches 0.5 when z reaches 0.

for name, predict in [("tree", tree_predict), ("logistic", logistic_predict)]:
    counts = confusion(rows["test"], predict)
    assert all(counts[key] == fixture["test"][name][key] for key in counts)
    correct = counts["tp"] + counts["tn"]
    print(f'{name.capitalize():<8} {counts} | accuracy {correct} ÷ {len(rows["test"])} = {correct / len(rows["test"]):.2%}')
    print(f'         Precision {counts["tp"] / (counts["tp"] + counts["fp"]):.2%} | recall {counts["tp"] / (counts["tp"] + counts["fn"]):.2%}')

## Continue in the Decision Trees lab

Use its smaller set of selected Spambase rows to reproduce a split by hand. The same mechanics extend to deeper trees, entropy, feature importance, and regression trees.

[Find the first split](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/#split) · [Trace a classification tree](https://sharanharsoor.github.io/ml-learning-lab/decision-trees/#tree)